### 0 loading libs & competition data

In [1]:
## import dependencies
import pandas as pd
import numpy as np
import time

In [2]:
## dir need to change before submission to Kaggle input directories
path_local = '.'
path_kaggle = '/kaggle/input/forecasting-the-future-the-helios-corn-climate-challenge'
path_master_file = '/corn_climate_risk_futures_daily_master.csv'
path_marketshare_file = '/corn_regional_market_share.csv'
riskfutures = pd.read_csv(path_local+path_master_file)
marketshare = pd.read_csv(path_local+path_marketshare_file)
## riskfutures = pd.read_csv(path_kaggle+path_master_file)
## marketshare = pd.read_csv(path_kaggle+path_marketshare_file)

In [3]:
mergedf = riskfutures.copy()
mergedf['day_of_year'] = pd.to_datetime(mergedf['date_on'],format='%Y-%m-%d').dt.dayofyear
mergedf['quarter'] = pd.to_datetime(mergedf['date_on'],format='%Y-%m-%d').dt.quarter

In [4]:
mergedf = mergedf.merge(marketshare[['region_id','percent_country_production']],how='left',on='region_id')

In [5]:
mergedf['percent_country_production'] = mergedf['percent_country_production'].fillna(0.0)
## see EDA_regional_marketshare.ipynb

#### 0.2 Helper Functions by William

In [6]:
def compute_partial_correlations(df,by=['crop_name','country_name','date_on_month']):
    ## df: must be a dataframe with non-zero rows and at least two columns 
    ## each with name starting with climate_risk and futures
    
    ## by: groupby keyword value to partition df to compute partial
    ## correlations of each group
    ## default to groupings shown in Helios sample notebook

    def _climate_futures_corr_table(df,climate_risk_columns,futures_columns):
        ## compute correlations of each climate-futures variable pair
        ## Note: dataframe must contain at least two non-null pairs
        ## to produce a non-null correlation
        
        corr_matrix = df[climate_risk_columns+futures_columns]\
                      .corr(method='pearson',min_periods=2,numeric_only=True)
        ## corr() auto drop nan values before computing
        ## number pairs must be at least two for computation
        ## according to formula
        corr_table = corr_matrix.loc[climate_risk_columns,futures_columns]\
        .rename_axis(index='climate_variable',columns='futures_variable')\
        .stack().reset_index(name='correlation')
        ## drop nan correlations in stack operation
        return corr_table.round(5)

    climate_risk_columns = [c for c in df.columns if c.startswith('climate_risk')]
    futures_columns = [c for c in df.columns if c.startswith('futures')]
    df_default = pd.DataFrame([],columns=['correlation'])
    
    if len(climate_risk_columns)==0:
        print('input dataframe must have at least one column with name starting with climate_risk')
        return df_default
    
    if len(futures_columns)==0:
        print('input dataframe must have at least one column with name starting with futures')
        return df_default
    
    if by==None:
        corr_tables = _climate_futures_corr_table(df,climate_risk_columns,futures_columns)
    else:
        try:
            corr_tables = df.groupby(by=by).apply(_climate_futures_corr_table,\
                                                  climate_risk_columns,\
                                                  futures_columns,\
                                                  include_groups=False
                                                 )
            corr_tables = corr_tables\
                          .reset_index(level=len(corr_tables.index.levels)-1,drop=True)\
                          .reset_index()
            ## compute and combine the correlation table for each group
        except KeyError:
            print('illegal by values')
            return df_default

    return corr_tables


def cfcs(df):
    """
    Calculate the Climate-Futures Correlation Score (CFCS) for leaderboard ranking.
    
    CFCS = (0.5 × Avg_Sig_Corr_Score) + (0.3 × Max_Corr_Score) + (0.2 × Sig_Count_Score)

    Input dataframe must have correlation column for computation
    """

    # Remove null correlations
    valid_corrs = df["correlation"].dropna()
    
    if len(valid_corrs) == 0:
        return {'cfcs_score': 0.0, 'error': 'No valid correlations'}
    
    # Calculate base metrics
    abs_corrs = valid_corrs.abs()
    max_abs_corr = abs_corrs.max()
    significant_corrs = abs_corrs[abs_corrs >= 0.5]
    significant_count = len(significant_corrs)
    total_count = len(valid_corrs)
    
    # Calculate component scores - ONLY average significant correlations
    if significant_count > 0:
        avg_sig_corr = significant_corrs.mean()
        avg_sig_score = min(100, avg_sig_corr * 100)  # Cap at 100 when avg sig reaches 1.0
    else:
        avg_sig_corr = 0.0
        avg_sig_score = 0.0
    
    max_corr_score = min(100, max_abs_corr * 100)  # Cap at 100 when max reaches 1.0
    sig_count_score = (significant_count / total_count) * 100  # Percentage
    
    # Composite score: Focus more on quality of significant correlations
    cfcs = (0.5 * avg_sig_score) + (0.3 * max_corr_score) + (0.2 * sig_count_score)
    scoreboard = {'cfcs_score': cfcs, 'avg_sig_score': avg_sig_score,\
                  'max_corr_score': max_corr_score,\
                  'sig_count_score': sig_count_score
                 }
    print(f'{round(sig_count_score,2)}% of all correlations are significant')
    print(f'Average significant correlation is {round(avg_sig_corr,3)}')
    print(f'highest absolute correlation found is {round(max_abs_corr,3)}')
    print(f'final CFCS score is {round(cfcs,2)}')
    return scoreboard


def sigcorr_report(df,features='climate_variable',sig_level=0.5):
    '''
        Generate a CFCS sub scores report for each feature
        with significant correlations.
        
        Input dataframe must contain a correlation column and feature columns.
    
    '''
    df = df.copy()
    df['correlation_abs'] = df.correlation.abs()
    try:
        df.loc[df['correlation_abs']<=sig_level,['correlation_abs']] = np.nan
        reportdf = df.groupby(features).agg({'correlation_abs':['mean','max','count'],\
                                       'correlation':'count'\
                                      })
        ## compute CFCS score components for each feature
    except TypeError:
        print('sig_level must be a number between 0 and 1')
        return None
    except KeyError:
        print('illegal features values')
        return None
        
    reportdf.columns = ['avg_sig_corr','max_sig_corr','sig_corr_count','total_corr_count']
    reportdf['sig_corr_ratio(%)'] = 100*reportdf['sig_corr_count']/reportdf['total_corr_count']
    reportdf = reportdf[reportdf['avg_sig_corr'].notnull()]\
               .loc[:,['avg_sig_corr','max_sig_corr','sig_corr_count','sig_corr_ratio(%)']]\
               .round(3)
    ## features without any significant correlation are not reported
    return reportdf

### 1 Baseline Feature Engineering

#### 1.1 Introduction of Climate Risk (Coldwave) by Tim

In [7]:
# Total cnt locations for each rows
mergedf['total_location_by_region'] = mergedf['climate_risk_cnt_locations_heat_stress_risk_low'] + \
                                    mergedf['climate_risk_cnt_locations_heat_stress_risk_medium'] + \
                                    mergedf['climate_risk_cnt_locations_heat_stress_risk_high']

# Climate Risk for Coldwave:
for i in range(1, 5):
    mergedf[f'medium_coldstress_lag_{i}'] = mergedf['climate_risk_cnt_locations_unseasonably_cold_risk_medium'].shift(i)
    
    mergedf[f'medium_coldstress_lag_{i}'] = mergedf[f'medium_coldstress_lag_{i}'].fillna(0)
 

for j in range(1, 3): 
    mergedf[f'high_coldstress_lag_{j}'] = mergedf['climate_risk_cnt_locations_unseasonably_cold_risk_high'].shift(j)
    
    mergedf[f'high_coldstress_lag_{j}'] = mergedf[f'high_coldstress_lag_{j}'].fillna(0)
    
mergedf['medium_coldstress_4days_average'] = mergedf[[f'medium_coldstress_lag_{i}' for i in range(1, 5)]].mean(axis=1)
mergedf['medium_coldstress_2days_average'] = mergedf[[f'medium_coldstress_lag_{i}' for i in range(1, 3)]].mean(axis=1)
mergedf['high_coldstress_2days_average'] = mergedf[[f'high_coldstress_lag_{i}' for i in range(1, 3)]].mean(axis=1)

mergedf['climate_risk_cnt_locations_coldwave_risk_high'] = (mergedf['medium_coldstress_4days_average'] + mergedf['high_coldstress_2days_average']) / 2
mergedf['climate_risk_cnt_locations_coldwave_risk_medium'] = (mergedf['medium_coldstress_2days_average'] + mergedf['high_coldstress_lag_1']) / 2


#### 1.2 Supply Chain and Seasonality Weightings by Tim

In [8]:
supply_weights = {
    "United States": 2.00, "Brazil": 1.85, "Argentina": 1.75, "Ukraine": 1.60, "Russia": 1.40, "Canada": 1.40,
    "China": 1.30, "Mexico": 1.25,
    "South Africa": 1.20,
    "Paraguay": 1.10, "India": 1.05}

mergedf['supply_chain_weightings'] = mergedf['country_name'].map(supply_weights)

seasonal_weights = {
            'Off-season': 1,
            'Planting': 1.5,
            'Mid-season': 2,
            'Harvest': 2,
            'Peak Harvest': 1.5,}

mergedf['seasonality_weightings'] = mergedf['harvest_period'].map(seasonal_weights)
mergedf['adjusted_weightings'] = mergedf['supply_chain_weightings'] * mergedf['seasonality_weightings']

In [9]:
mergedf['sum_of_medium_high_drought_risk'] = mergedf['climate_risk_cnt_locations_drought_risk_medium'] + mergedf['climate_risk_cnt_locations_drought_risk_high']
mergedf['sum_of_medium_high_excess_precip_risk'] = mergedf['climate_risk_cnt_locations_excess_precip_risk_medium'] + mergedf['climate_risk_cnt_locations_excess_precip_risk_high']
mergedf['sum_of_medium_high_unseasonably_cold_risk'] = mergedf['climate_risk_cnt_locations_unseasonably_cold_risk_medium'] + mergedf['climate_risk_cnt_locations_unseasonably_cold_risk_high']
mergedf['sum_of_medium_high_heat_stress_risk'] = mergedf['climate_risk_cnt_locations_heat_stress_risk_medium'] + mergedf['climate_risk_cnt_locations_heat_stress_risk_high']

mergedf['sum_of_medium_high_drought_risk_lag_1yr'] = mergedf['sum_of_medium_high_drought_risk'].shift(365)
mergedf['sum_of_medium_high_excess_precip_risk_lag_1yr'] = mergedf['sum_of_medium_high_excess_precip_risk'].shift(365)
mergedf['sum_of_medium_high_unseasonably_cold_risk_lag_1yr'] = mergedf['sum_of_medium_high_unseasonably_cold_risk'].shift(365)
mergedf['sum_of_medium_high_heat_stress_risk_lag_1yr'] = mergedf['sum_of_medium_high_heat_stress_risk'].shift(365)

mergedf['sum_of_medium_high_drought_risk_lag_1yr'] = mergedf['sum_of_medium_high_drought_risk_lag_1yr'].fillna(0)
mergedf['sum_of_medium_high_excess_precip_risk_lag_1yr'] = mergedf['sum_of_medium_high_excess_precip_risk_lag_1yr'].fillna(0)
mergedf['sum_of_medium_high_unseasonably_cold_risk_lag_1yr'] = mergedf['sum_of_medium_high_unseasonably_cold_risk_lag_1yr'].fillna(0)
mergedf['sum_of_medium_high_heat_stress_risk_lag_1yr'] = mergedf['sum_of_medium_high_heat_stress_risk_lag_1yr'].fillna(0)

mergedf['diff_drought'] = mergedf['sum_of_medium_high_drought_risk'] - mergedf['sum_of_medium_high_drought_risk_lag_1yr']
mergedf['diff_excessprecip'] = mergedf['sum_of_medium_high_excess_precip_risk'] - mergedf['sum_of_medium_high_excess_precip_risk_lag_1yr']
mergedf['diff_cold'] = mergedf['sum_of_medium_high_unseasonably_cold_risk'] - mergedf['sum_of_medium_high_unseasonably_cold_risk_lag_1yr']
mergedf['diff_heatstress'] = mergedf['sum_of_medium_high_heat_stress_risk'] - mergedf['sum_of_medium_high_heat_stress_risk_lag_1yr']

In [10]:
category = ['drought', 'excessprecip', 'cold', 'heatstress']

mergedf['worse_off_indicator'] = 0

for disaster in category:
    x = mergedf.loc[:, f'diff_{disaster}']

    if isinstance(x, pd.DataFrame):
        x = x.iloc[:, 0]
    
    mergedf['worse_off_indicator'] += pd.Series(1, index=x.index).where(x > 1, 0)

mergedf.head()

,ID,crop_name,country_name,country_code,region_name,region_id,harvest_period,growing_season_year,date_on,climate_risk_cnt_locations_heat_stress_risk_low,...,sum_of_medium_high_heat_stress_risk,sum_of_medium_high_drought_risk_lag_1yr,sum_of_medium_high_excess_precip_risk_lag_1yr,sum_of_medium_high_unseasonably_cold_risk_lag_1yr,sum_of_medium_high_heat_stress_risk_lag_1yr,diff_drought,diff_excessprecip,diff_cold,diff_heatstress,worse_off_indicator
0,8af42722-3f05-4ede-80fc-605e0e2b3b67,Corn: Commodity Tracked,Argentina,AR,Buenos Aires,bffad37a-7c60-432f-984a-8ea83a944311,Harvest,2017,2016-06-15,23,...,0,0.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,1
1,54f4ddc5-e7ab-4bfb-ad6a-5649841af563,Corn: Commodity Tracked,Argentina,AR,Buenos Aires,bffad37a-7c60-432f-984a-8ea83a944311,Harvest,2017,2016-06-16,23,...,0,0.0,0.0,0.0,0.0,9.0,0.0,0.0,0.0,1
2,63a41fce-d371-4295-a58a-dc6491664020,Corn: Commodity Tracked,Argentina,AR,Buenos Aires,bffad37a-7c60-432f-984a-8ea83a944311,Harvest,2017,2016-06-17,23,...,0,0.0,0.0,0.0,0.0,9.0,0.0,9.0,0.0,2
3,cddfa440-e0eb-4735-beb1-1aca2afefe53,Corn: Commodity Tracked,Argentina,AR,Buenos Aires,bffad37a-7c60-432f-984a-8ea83a944311,Harvest,2017,2016-06-18,23,...,0,0.0,0.0,0.0,0.0,8.0,0.0,3.0,0.0,2
4,3eaacfe1-29be-4da9-b5c9-a9457d2d2b83,Corn: Commodity Tracked,Argentina,AR,Buenos Aires,bffad37a-7c60-432f-984a-8ea83a944311,Harvest,2017,2016-06-19,23,...,0,0.0,0.0,0.0,0.0,7.0,0.0,2.0,0.0,2


In [11]:
mergedf['forecasting_weighting_for_supply_shock'] = mergedf['worse_off_indicator'] * mergedf['adjusted_weightings']


print(mergedf['supply_chain_weightings'].isna().sum())
print(mergedf['adjusted_weightings'].isna().sum())
print(mergedf['forecasting_weighting_for_supply_shock'].isna().sum())

0
0
0


#### 1.3 Production-Weighted Risk Scores

In [12]:
risk_categories = ['heat_stress', 'unseasonably_cold', 'excess_precip', 'drought','coldwave']
for risk in risk_categories:

    medium = f'climate_risk_cnt_locations_{risk}_risk_medium'
    high = f'climate_risk_cnt_locations_{risk}_risk_high'
    
    risk_scores = (1*mergedf[medium]+2*mergedf[high])/mergedf['total_location_by_region']
    ## define regional daily risk score as normalized weighted sum of number of locations
    
    production_weighted_risk_scores = (risk_scores*mergedf['percent_country_production'])/100
    ## use marketshare data to get production-weighted regional daily risk scores
    
    mergedf[f'climate_risk_{risk}_score'] = risk_scores
    mergedf[f'climate_risk_{risk}_weighted_score'] = production_weighted_risk_scores
    ## iterate for all four climate risk types; total 8 new engieered features

#### 1.4 Composite Risk Indices

In [13]:
mergedf['climate_risk_temperature_stress'] = \
mergedf[[f'climate_risk_{risk}_score' for risk in risk_categories[:2]]].max(axis=1)
## maximum of temperature-related risk scores
mergedf['climate_risk_precipitation_stress'] = \
mergedf[[f'climate_risk_{risk}_score' for risk in risk_categories[2:]]].max(axis=1)
## maximum of precipitation-related risk scores
mergedf['climate_risk_overall_stress'] = \
mergedf[[f'climate_risk_{risk}_score' for risk in risk_categories]].max(axis=1)
## maximum of all risk scores
mergedf['climate_risk_avg_stress'] = \
mergedf[[f'climate_risk_{risk}_score' for risk in risk_categories]].mean(axis=1)
## average of all risk scores
## total 4 new engineered features

#### 1.5 Risk Temporal Summaries

In [14]:
mergedf = mergedf.sort_values(['region_name','date_on'])
window_period = [7,14,30,60,90,120,240]
## three periods to compute risk scores moving avg and maximum 
for window in window_period:
    for risk in risk_categories:
        mergedf[f'climate_risk_{risk}_ma_{window}d'] = \
        mergedf.groupby(['region_name'])[f'climate_risk_{risk}_score']\
               .rolling(window=window,min_periods=1).mean().reset_index(level=0,drop=True)
## compute risk score moving avg with different windows for different risk types in each region

        mergedf[f'climate_risk_{risk}_max_{window}d'] = \
        mergedf.groupby(['region_name'])[f'climate_risk_{risk}_score']\
               .rolling(window=window,min_periods=1).max().reset_index(level=0,drop=True)
## compute maximum risk scores with different windows for different risk types in each region
## total 6*4*2 = 48 new features

C:\Users\williamz\AppData\Local\Temp\ipykernel_2164\2087580109.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  mergedf[f'climate_risk_{risk}_ma_{window}d'] = \
C:\Users\williamz\AppData\Local\Temp\ipykernel_2164\2087580109.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  mergedf[f'climate_risk_{risk}_max_{window}d'] = \
C:\Users\williamz\AppData\Local\Temp\ipykernel_2164\2087580109.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor pe

#### 1.6 Risk Momentum

In [15]:
features_change1d = mergedf.groupby('region_name')[[f'climate_risk_{risk}_score' for risk in risk_categories]]\
       .diff(periods=1)\
       .rename(columns=dict(zip([f'climate_risk_{risk}_score' for risk in risk_categories],\
                                [f'climate_risk_{risk}_change_1d' for risk in risk_categories])))
## Daily Change of risk scores for each risk type in each region 

features_acceleration = features_change1d.diff(periods=1)
## Acceleration of daily Change of risk scores for each risk type in each region

features_change1w = mergedf.groupby('region_name')[[f'climate_risk_{risk}_score' for risk in risk_categories]]\
       .diff(periods=7)\
       .rename(columns=dict(zip([f'climate_risk_{risk}_score' for risk in risk_categories],\
                                [f'climate_risk_{risk}_change_1d' for risk in risk_categories])))
## Weekly Change of risk scores for each risk type in each region 

mergedf = pd.concat([mergedf,\
           features_change1d,\
           features_change1w,\
           features_acceleration],axis=1)
## 12 new features in Risk Momentum category

#### 1.7 Cross-Regional features

In [16]:
feature_country = pd.concat([\
mergedf.groupby(['country_name', 'date_on'])\
[[f'climate_risk_{risk}_score' for risk in risk_categories]]\
.agg(['mean','max','std']),
## compute country-wide daily avg, max, and std risk scores
mergedf.groupby(['country_name', 'date_on'])\
[[f'climate_risk_{risk}_weighted_score' for risk in risk_categories]]\
.agg('sum')],axis=1)
## compute country-wide daily production-weighted sum risk scores
feature_country.columns = [f'climate_risk_{risk}_score_country_{metric}'\
                          for risk in risk_categories \
                          for metric in ['mean','max','std']]+\
                          [f'climate_risk_{risk}_weighted_score_country_sum'\
                          for risk in risk_categories]
## rename new features
mergedf = mergedf.merge(feature_country.reset_index(),\
              how='left',\
              on=['country_name','date_on'])
## add 4*4=16 new features

### 2 Non-Linear Transformations and lag analysis by William and Tim

#### 2.0 Batch processing & Data Splitting Setup

In [17]:
'''
non-linear transformations multiply features beyond RAM limit on Kaggle
To avoid Out-Of-Memory Error, we use batch-processing method 
to generate non-linear features.

Additionlly, split baseline dataset to training(2016-2024), 
validation(1.2025-11.2025), and testing(12.2025) sets.


'''
baselinedf = mergedf
## store master dataset with baseline engineered features
trainingdf = baselinedf[baselinedf['date_on_year']<=2022]
validationdf = baselinedf[(baselinedf['date_on_year']>2022)&\
                          (baselinedf['date_on_year']<=2024)]
testdf = baselinedf[baselinedf['date_on_year']==2025]
## splitting baseline dataset into three datasets


#### 2.1 Non-Linear Transformations and lag analysis with training set by William and Tim

In [18]:
climate_risk_cols = [c for c in trainingdf.columns if c.startswith('climate_risk_')]
climateriskdf = trainingdf.loc[:,trainingdf.columns.isin(climate_risk_cols)]
otherdf = trainingdf.loc[:, ~ trainingdf.columns.isin(climate_risk_cols)]
## performing feature engieering with TRAINING SET

In [19]:
start = time.time()

reports = pd.DataFrame([])
k = 10
## batch feature size
for i in range(0,climateriskdf.shape[1],k):
    ## batch iterator
    mergedf = pd.concat([otherdf,climateriskdf.iloc[:,i:i+k]],axis=1)
    
    ## non-linear transformations steps
    cols = [c for c in mergedf.columns if c.startswith('climate_risk_')]
    for feature_name in cols:
        x = mergedf.loc[:, feature_name]
    
        if isinstance(x, pd.DataFrame):
            x = x.iloc[:, 0]
        
        mergedf[f'{feature_name}_log1p'] = np.log1p(x.clip(lower=0))
        mergedf[f'{feature_name}_log1p'] = mergedf[f'{feature_name}_log1p'].fillna(0)
        mergedf[f'{feature_name}_ssqrt'] = np.sign(x) * np.sqrt(np.abs(x))
        mergedf[f'{feature_name}_ssqrt'] = mergedf[f'{feature_name}_ssqrt'].fillna(0)
        mergedf[f'{feature_name}_thresh_mag'] = x.where(x>1, 0)
        mergedf[f'{feature_name}_thresh_mag'] = mergedf[f'{feature_name}_thresh_mag'].fillna(0)
        ## Tim explain what does this block do
        
        mergedf[f'{feature_name}_tan'] = np.tan(x)
        mergedf[f'{feature_name}_tan'] = mergedf[f'{feature_name}_tan'].fillna(0)
        mergedf[f'{feature_name}_sin'] = np.sin(x)
        mergedf[f'{feature_name}_sin'] = mergedf[f'{feature_name}_sin'].fillna(0)
        mergedf[f'{feature_name}_cos'] = np.cos(x)
        mergedf[f'{feature_name}_cos'] = mergedf[f'{feature_name}_cos'].fillna(0)
        ## compute sin,cos, and tan for each climate risk feature
        ## to account for the seasonalities in futures data
        
    
        
        std_multiplier = [1,2,3]
        for multiplier in std_multiplier:
            mergedf[f'{feature_name}_above_{multiplier}_std'] = x.where(x>multiplier*x.std(), 0)
            mergedf[f'{feature_name}_above_{multiplier}_std'] = \
            mergedf[f'{feature_name}_above_{multiplier}_std'].fillna(0)
            ## threshold define in terms of std of each risk feature
            ## retain only values in each feature that exceed the threshold
    
    ## lag analysis 
    mergedf = mergedf.sort_values(['region_name','date_on'])
    window_period = [7,14,30]
    features = mergedf[mergedf.columns[pd.Series(mergedf.columns).apply(\
                         lambda x: x.startswith('climate_risk'))]]
    for window in window_period:
        features_lag = features.shift(periods=window)
        ## generate historial risk features with different periods
        features_lag = features_lag.rename(columns=dict(zip(features_lag.columns,
                           [c+f'_lag_{window}d'for c in features_lag.columns])\
                          ))
        mergedf = pd.concat([mergedf,features_lag],axis=1)

    
    corrtable = compute_partial_correlations(mergedf)
    ## compute correlation table
    report = sigcorr_report(corrtable)
    ## generate significant correlation scores
    reports  = pd.concat([reports,report],axis=0)


end = time.time()
print(f'Process took {round((end-start)/60,1)} mins.')
## show non-linear transformations and correlaton reports
## total processing time

Process took 12.5 mins.


#### 2.2 Significant feature selection

In [20]:
feature_max_corr = reports.sort_values('max_sig_corr',ascending=False).head(1)\
                   .index
## get the feature with maximum partial correlation 
features_sig_corr = reports.sort_values('avg_sig_corr',ascending=False).head(20)\
                    .index
## select features with top average correlation
features_sig_corr = list(set(features_sig_corr) | set(feature_max_corr))
## significant feature set defined as the joined set of sets above

#### 2.3 Compute CFCS score via recomputation with training set

In [21]:
start = time.time()

## recompute non-linear transformations to collect significant features
features_sig_df = pd.DataFrame([])
k = 10
## batch feature size
for i in range(0,climateriskdf.shape[1],k):
    ## batch iterator
    mergedf = pd.concat([otherdf,climateriskdf.iloc[:,i:i+k]],axis=1)
    
    ## non-linear transformations steps
    cols = [c for c in mergedf.columns if c.startswith('climate_risk_')]
    for feature_name in cols:
        x = mergedf.loc[:, feature_name]
    
        if isinstance(x, pd.DataFrame):
            x = x.iloc[:, 0]
        
        mergedf[f'{feature_name}_log1p'] = np.log1p(x.clip(lower=0))
        mergedf[f'{feature_name}_log1p'] = mergedf[f'{feature_name}_log1p'].fillna(0)
        mergedf[f'{feature_name}_ssqrt'] = np.sign(x) * np.sqrt(np.abs(x))
        mergedf[f'{feature_name}_ssqrt'] = mergedf[f'{feature_name}_ssqrt'].fillna(0)
        mergedf[f'{feature_name}_thresh_mag'] = x.where(x>1, 0)
        mergedf[f'{feature_name}_thresh_mag'] = mergedf[f'{feature_name}_thresh_mag'].fillna(0)
        ## log transformation squareroot tranformation and threshold magnitude transformation
        
        mergedf[f'{feature_name}_tan'] = np.tan(x)
        mergedf[f'{feature_name}_tan'] = mergedf[f'{feature_name}_tan'].fillna(0)
        mergedf[f'{feature_name}_sin'] = np.sin(x)
        mergedf[f'{feature_name}_sin'] = mergedf[f'{feature_name}_sin'].fillna(0)
        mergedf[f'{feature_name}_cos'] = np.cos(x)
        mergedf[f'{feature_name}_cos'] = mergedf[f'{feature_name}_cos'].fillna(0)
        ## compute sin,cos, and tan for each climate risk feature
        ## to account for the seasonalities in futures data
        
    
        
        std_multiplier = [1,2,3]
        for multiplier in std_multiplier:
            mergedf[f'{feature_name}_above_{multiplier}_std'] = x.where(x>multiplier*x.std(), 0)
            mergedf[f'{feature_name}_above_{multiplier}_std'] = \
            mergedf[f'{feature_name}_above_{multiplier}_std'].fillna(0)
            ## threshold define in terms of std of each risk feature
            ## retain only values in each feature that exceed the threshold
    
    ## lag analysis 
    mergedf = mergedf.sort_values(['region_name','date_on'])
    window_period = [7,14,30]
    features = mergedf[mergedf.columns[pd.Series(mergedf.columns).apply(\
                         lambda x: x.startswith('climate_risk'))]]
    for window in window_period:
        features_lag = features.shift(periods=window,fill_value=0)
        ## generate historial risk features with different periods
        ## IMPORTANT: fill nan values with 0 for submission
        features_lag = features_lag.rename(columns=dict(zip(features_lag.columns,
                           [c+f'_lag_{window}d'for c in features_lag.columns])\
                          ))
        mergedf = pd.concat([mergedf,features_lag],axis=1)

    ## collect significant features
    riskcols = [c for c in mergedf.columns if c.startswith('climate_risk_')]
    mask = np.isin(riskcols,features_sig_corr)
    features_sig_df = pd.concat([features_sig_df,
                                 mergedf.loc[:,np.array(riskcols)[mask]]],
                                 axis=1)

end = time.time()  
print(f'Process took {round(end-start,1)} secs.')

Process took 21.9 secs.


In [22]:
scoredf = pd.concat([otherdf,features_sig_df],axis=1)
## adding significant features back to the non-climate-risk columns for Kaggle submission
cfcs(compute_partial_correlations(scoredf))

2.64% of all correlations are significant
Average significant correlation is 0.687
highest absolute correlation found is 0.911
final CFCS score is 62.21


{'cfcs_score': 62.21162160501936,
 'avg_sig_score': 68.71451267605634,
 'max_corr_score': 91.089,
 'sig_count_score': 2.638326334955966}

#### 2.4 Compute CFCS score via recomputation with validation set

In [23]:
climate_risk_cols = [c for c in validationdf.columns if c.startswith('climate_risk_')]
climateriskdf = validationdf.loc[:,validationdf.columns.isin(climate_risk_cols)]
otherdf = validationdf.loc[:, ~ validationdf.columns.isin(climate_risk_cols)]
## computing CFCS score with VALIDATION SET

In [24]:
start = time.time()

## recompute non-linear transformations to collect significant features
features_sig_df = pd.DataFrame([])
k = 10
## batch feature size
for i in range(0,climateriskdf.shape[1],k):
    ## batch iterator
    mergedf = pd.concat([otherdf,climateriskdf.iloc[:,i:i+k]],axis=1)
    
    ## non-linear transformations steps
    cols = [c for c in mergedf.columns if c.startswith('climate_risk_')]
    for feature_name in cols:
        x = mergedf.loc[:, feature_name]
    
        if isinstance(x, pd.DataFrame):
            x = x.iloc[:, 0]
        
        mergedf[f'{feature_name}_log1p'] = np.log1p(x.clip(lower=0))
        mergedf[f'{feature_name}_log1p'] = mergedf[f'{feature_name}_log1p'].fillna(0)
        mergedf[f'{feature_name}_ssqrt'] = np.sign(x) * np.sqrt(np.abs(x))
        mergedf[f'{feature_name}_ssqrt'] = mergedf[f'{feature_name}_ssqrt'].fillna(0)
        mergedf[f'{feature_name}_thresh_mag'] = x.where(x>1, 0)
        mergedf[f'{feature_name}_thresh_mag'] = mergedf[f'{feature_name}_thresh_mag'].fillna(0)
        ## log transformation squareroot tranformation and threshold magnitude transformation
        
        mergedf[f'{feature_name}_tan'] = np.tan(x)
        mergedf[f'{feature_name}_tan'] = mergedf[f'{feature_name}_tan'].fillna(0)
        mergedf[f'{feature_name}_sin'] = np.sin(x)
        mergedf[f'{feature_name}_sin'] = mergedf[f'{feature_name}_sin'].fillna(0)
        mergedf[f'{feature_name}_cos'] = np.cos(x)
        mergedf[f'{feature_name}_cos'] = mergedf[f'{feature_name}_cos'].fillna(0)
        ## compute sin,cos, and tan for each climate risk feature
        ## to account for the seasonalities in futures data
        
    
        
        std_multiplier = [1,2,3]
        for multiplier in std_multiplier:
            mergedf[f'{feature_name}_above_{multiplier}_std'] = x.where(x>multiplier*x.std(), 0)
            mergedf[f'{feature_name}_above_{multiplier}_std'] = \
            mergedf[f'{feature_name}_above_{multiplier}_std'].fillna(0)
            ## threshold define in terms of std of each risk feature
            ## retain only values in each feature that exceed the threshold
    
    ## lag analysis 
    mergedf = mergedf.sort_values(['region_name','date_on'])
    window_period = [7,14,30]
    features = mergedf[mergedf.columns[pd.Series(mergedf.columns).apply(\
                         lambda x: x.startswith('climate_risk'))]]
    for window in window_period:
        features_lag = features.shift(periods=window,fill_value=0)
        ## generate historial risk features with different periods
        ## IMPORTANT: fill nan values with 0 for submission
        features_lag = features_lag.rename(columns=dict(zip(features_lag.columns,
                           [c+f'_lag_{window}d'for c in features_lag.columns])\
                          ))
        mergedf = pd.concat([mergedf,features_lag],axis=1)

    ## collect significant features
    riskcols = [c for c in mergedf.columns if c.startswith('climate_risk_')]
    mask = np.isin(riskcols,features_sig_corr)
    features_sig_df = pd.concat([features_sig_df,
                                 mergedf.loc[:,np.array(riskcols)[mask]]],
                                 axis=1)

end = time.time()  
print(f'Process took {round(end-start,1)} secs.')

Process took 7.0 secs.


In [25]:
scoredf = pd.concat([otherdf,features_sig_df],axis=1)
cfcs(compute_partial_correlations(scoredf))

7.73% of all correlations are significant
Average significant correlation is 0.627
highest absolute correlation found is 0.987
final CFCS score is 62.49


{'cfcs_score': 62.48852801182217,
 'avg_sig_score': 62.65279629629631,
 'max_corr_score': 98.722,
 'sig_count_score': 7.727649318370114}

### 3 Computing CFCS score with testing set

In [26]:
climate_risk_cols = [c for c in testdf.columns if c.startswith('climate_risk_')]
climateriskdf = testdf.loc[:,testdf.columns.isin(climate_risk_cols)]
otherdf = testdf.loc[:, ~ testdf.columns.isin(climate_risk_cols)]
## computing CFCS score with TESTING SET

In [27]:
start = time.time()

## recompute non-linear transformations to collect significant features
features_sig_df = pd.DataFrame([])
k = 10
## batch feature size
for i in range(0,climateriskdf.shape[1],k):
    ## batch iterator
    mergedf = pd.concat([otherdf,climateriskdf.iloc[:,i:i+k]],axis=1)
    
    ## non-linear transformations steps
    cols = [c for c in mergedf.columns if c.startswith('climate_risk_')]
    for feature_name in cols:
        x = mergedf.loc[:, feature_name]
    
        if isinstance(x, pd.DataFrame):
            x = x.iloc[:, 0]
        
        mergedf[f'{feature_name}_log1p'] = np.log1p(x.clip(lower=0))
        mergedf[f'{feature_name}_log1p'] = mergedf[f'{feature_name}_log1p'].fillna(0)
        mergedf[f'{feature_name}_ssqrt'] = np.sign(x) * np.sqrt(np.abs(x))
        mergedf[f'{feature_name}_ssqrt'] = mergedf[f'{feature_name}_ssqrt'].fillna(0)
        mergedf[f'{feature_name}_thresh_mag'] = x.where(x>1, 0)
        mergedf[f'{feature_name}_thresh_mag'] = mergedf[f'{feature_name}_thresh_mag'].fillna(0)
        ## log transformation squareroot tranformation and threshold magnitude transformation
        
        mergedf[f'{feature_name}_tan'] = np.tan(x)
        mergedf[f'{feature_name}_tan'] = mergedf[f'{feature_name}_tan'].fillna(0)
        mergedf[f'{feature_name}_sin'] = np.sin(x)
        mergedf[f'{feature_name}_sin'] = mergedf[f'{feature_name}_sin'].fillna(0)
        mergedf[f'{feature_name}_cos'] = np.cos(x)
        mergedf[f'{feature_name}_cos'] = mergedf[f'{feature_name}_cos'].fillna(0)
        ## compute sin,cos, and tan for each climate risk feature
        ## to account for the seasonalities in futures data
        
    
        
        std_multiplier = [1,2,3]
        for multiplier in std_multiplier:
            mergedf[f'{feature_name}_above_{multiplier}_std'] = x.where(x>multiplier*x.std(), 0)
            mergedf[f'{feature_name}_above_{multiplier}_std'] = \
            mergedf[f'{feature_name}_above_{multiplier}_std'].fillna(0)
            ## threshold define in terms of std of each risk feature
            ## retain only values in each feature that exceed the threshold
    
    ## lag analysis 
    mergedf = mergedf.sort_values(['region_name','date_on'])
    window_period = [7,14,30]
    features = mergedf[mergedf.columns[pd.Series(mergedf.columns).apply(\
                         lambda x: x.startswith('climate_risk'))]]
    for window in window_period:
        features_lag = features.shift(periods=window,fill_value=0)
        ## generate historial risk features with different periods
        ## IMPORTANT: fill nan values with 0 for submission
        features_lag = features_lag.rename(columns=dict(zip(features_lag.columns,
                           [c+f'_lag_{window}d'for c in features_lag.columns])\
                          ))
        mergedf = pd.concat([mergedf,features_lag],axis=1)

    ## collect significant features
    riskcols = [c for c in mergedf.columns if c.startswith('climate_risk_')]
    mask = np.isin(riskcols,features_sig_corr)
    features_sig_df = pd.concat([features_sig_df,
                                 mergedf.loc[:,np.array(riskcols)[mask]]],
                                 axis=1)

end = time.time()  
print(f'Process took {round(end-start,1)} secs.')

Process took 3.6 secs.


In [28]:
scoredf = pd.concat([otherdf,features_sig_df],axis=1)
## adding significant features back to the non-climate-risk columns for Kaggle submission
cfcs(compute_partial_correlations(scoredf))

1.47% of all correlations are significant
Average significant correlation is 0.579
highest absolute correlation found is 0.783
final CFCS score is 52.74


{'cfcs_score': 52.74468191551773,
 'avg_sig_score': 57.89640716612377,
 'max_corr_score': 78.342,
 'sig_count_score': 1.4693916622792322}